<a href="https://colab.research.google.com/github/rama603/Reading-Sign-Language-/blob/main/Reading_Sign_Language_stage3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Set seed for exact reproducibility across stages
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# 1. Load Data
train_df = pd.read_csv("sign_mnist_train.csv")
test_df = pd.read_csv("sign_mnist_test.csv")

X_train_full = train_df.drop(columns="label").values.astype("float32") / 255.0
y_train_full = train_df["label"].values

X_test = test_df.drop(columns="label").values.astype("float32") / 255.0
y_test = test_df["label"].values

# 2. Remap labels to contiguous range 0 to 23 (ignoring J=9 and Z=25)
unique_labels = sorted(np.unique(y_train_full))
label_map = {old_id: new_id for new_id, old_id in enumerate(unique_labels)}
inv_label_map = {new_id: old_id for new_id, old_id in enumerate(unique_labels)}

y_train_mapped = np.array([label_map[y] for y in y_train_full])
y_test_mapped = np.array([label_map[y] for y in y_test])

# 3. Maintain Exact Same Validation Split as Stages 1 & 2
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_mapped, test_size=0.2, random_state=SEED, stratify=y_train_mapped
)

# Reshape into PyTorch image tensor format: (Batch, Channels, Height, Width) -> (N, 1, 28, 28)
X_train_t = torch.tensor(X_train).view(-1, 1, 28, 28)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_val_t = torch.tensor(X_val).view(-1, 1, 28, 28)
y_val_t = torch.tensor(y_val, dtype=torch.long)

X_test_t = torch.tensor(X_test).view(-1, 1, 28, 28)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)

print(f"Data prepared! Number of classes: {len(unique_labels)}")
print(f"Training batches: {len(train_loader)}, Validation batches: {len(val_loader)}")

# 4. Define the Fixed CNN Architecture Required by Project Instructions
class SignLanguageCNN(nn.Module):
    def __init__(self, num_classes=24):
        super(SignLanguageCNN, self).__init__()
        # Conv Layer 1: 1 -> 16 filters, 3x3 kernel, padding=1
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Conv Layer 2: 16 -> 32 filters, 3x3 kernel, padding=1
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Softmax classifier layer on top of extracted features
        # Image size after two 2x2 max pools: 28 -> 14 -> 7. Feature map size: 32 * 7 * 7 = 1568
        self.fc = nn.Linear(32 * 7 * 7, num_classes)

    def forward(self, x, return_features=False):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))

        # Flatten features
        features = x.view(x.size(0), -1)

        if return_features:
            return features

        out = self.fc(features)
        return out

model = SignLanguageCNN(num_classes=len(unique_labels))
print(model)

Data prepared! Number of classes: 24
Training batches: 344, Validation batches: 86
SignLanguageCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=1568, out_features=24, bias=True)
)
